# CodeTune v2 — 01 Data Preparation

**Environment**: Local (RTX 5070 8GB)  
**Goal**: Download datasets, filter/clean/dedup SFT data, generate targeted failure data with MiniMax API, upload to HF Hub.

**Run order**:
1. Cell 1: Verify environment & dependencies
2. Cell 2: Load `.env` and test MiniMax API
3. Cell 3: Download raw datasets
4. Cell 4: Run `prepare_sft_data.py`
5. Cell 5: Run `generate_targeted_data.py`
6. Cell 6: Run `data_stats.py` — quality report
7. Cell 7: Upload processed data to HuggingFace Hub

In [6]:
# Cell 1: Verify environment
import sys, subprocess
print(f'Python: {sys.version}')

required = ['datasets', 'datasketch', 'openai', 'transformers', 'tenacity']
for pkg in required:
    try:
        __import__(pkg.replace('-', '_'))
        print(f'  {pkg}: OK')
    except ImportError:
        print(f'  {pkg}: MISSING — run: uv sync')

Python: 3.14.3 (main, Feb  3 2026, 22:53:56) [MSC v.1944 64 bit (AMD64)]
  datasets: OK
  datasketch: OK
  openai: OK
  transformers: OK
  tenacity: OK


In [7]:
# Cell 2: Load .env and test MiniMax API
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load from project root .env
env_path = Path('..') / '.env'
load_dotenv(env_path)

api_key  = os.environ.get('MINIMAX_API_KEY')
base_url = os.environ.get('MINIMAX_BASE_URL', 'https://api.minimax.io/v1')

assert api_key, 'MINIMAX_API_KEY not set! Copy .env.example to .env and fill in your key.'
print(f'API key: {api_key[:8]}...{api_key[-4:]}')
print(f'Base URL: {base_url}')

# Quick connectivity test
client = OpenAI(api_key=api_key, base_url=base_url)
try:
    resp = client.chat.completions.create(
        model='MiniMax-M2.5',
        messages=[{'role': 'user', 'content': 'Reply OK'}],
        max_tokens=10,
    )
    print(f'API test: {resp.choices[0].message.content}')
except Exception as e:
    print(f'API test FAILED: {e}')
    print('Check your MINIMAX_API_KEY and MINIMAX_BASE_URL')

API key: [REDACTED]
Base URL: https://api.minimax.io/v1
API test: <think>
The user just said "Reply OK".
</think>




In [8]:
# Cell 3: Download raw datasets (cache to HuggingFace cache dir)
# This may take 5-15 minutes depending on connection speed
from datasets import load_dataset

print('Downloading Magicoder-OSS-Instruct-75K...')
magicoder = load_dataset('ise-uiuc/Magicoder-OSS-Instruct-75K', split='train', trust_remote_code=True)
print(f'  Magicoder: {len(magicoder):,} samples')

print('Downloading evol-codealpaca-v1...')
evol = load_dataset('theblackcat102/evol-codealpaca-v1', split='train', trust_remote_code=True)
print(f'  Evol: {len(evol):,} samples')

print('Downloading HumanEval...')
humaneval = load_dataset('openai/openai_humaneval', split='test', trust_remote_code=True)
print(f'  HumanEval: {len(humaneval)} problems')

print('\nAll datasets downloaded and cached.')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ise-uiuc/Magicoder-OSS-Instruct-75K' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'theblackcat102/evol-codealpaca-v1' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Magicoder: 75,197 samples


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openai/openai_humaneval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Evol: 111,272 samples
  HumanEval: 164 problems

All datasets downloaded and cached.


In [ ]:
# Cell 4: Prepare SFT data
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, '../scripts/prepare_sft_data.py',
     '--sources', 'magicoder,evol,humaneval',
     '--output-dir', '../data/processed'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')

In [ ]:
# Cell 5: Generate targeted data with MiniMax API
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, '../scripts/generate_targeted_data.py',
     '--target-per-pattern', '50',
     '--batch-size', '5',
     '--model', 'MiniMax-M2.5'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')

In [ ]:
# Cell 6: Data quality report
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, '../scripts/data_stats.py',
     '--data-dir', '../data/processed'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')

In [ ]:
# Cell 8: Generate DPO pairs locally (RTX 5070 4-bit inference + MiniMax scoring)
# Output: data/processed/dpo_pairs.jsonl + dpo_pairs_val.jsonl
# Upload these two files to Google Drive: codetune/dpo_pairs/ before running Colab Cell 5
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, '../scripts/generate_dpo_pairs.py',
     '--sft-checkpoint', 'Michlitt/codetune-v2-sft-C',
     '--model', 'MiniMax-M2.5'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding='utf-8', errors='replace', bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W0317 15:11:22.772000 7080 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Local path not found — downloading from HF Hub: Michlitt/codetune-v2-sft-C

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 63872.65it/s]
Downloaded to E:\Data\Codetune\hf_cache\hub\models--Michlitt--codetune-v2-sft-C\snapshots\7a5a36a3b0758d810ed915a6ce9ded9bf4910962
Loading SFT model from E:\Data\Codetune\hf_cache\hub\models--Michlitt--codetune-v2-sft-C\snapshots\7a5a36a3b0758d810ed915a6ce9ded9bf4910962 (4-bit) …
==((====))==  Unsloth 2026.3.5: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 5070 Laptop GPU. Num GPUs = 1. Max